In [ ]:
!pip install aiohttp aiofiles beautifulsoup4 pandas

In [10]:
import asyncio
import io
import aiohttp
import csv
from bs4 import BeautifulSoup
import time
import random
import pandas as pd

# Set to "simple" (cookie injection) or "real" (full 3-step handshake)
MODE = "real"

# --- ⚙️ CONFIGURATION ⚙️ ---
TOURNAMENTS = [
    # {"id": "4626469", "name": "PBL", "date": "09/06/2025"},
    # {"id": "9460497", "name": "UBL 1-1", "date": "11/08/2025"},
    {"id": "608123", "name": "UBL 1-2", "date": "11/15/2025"},
    # {"id": "8762799", "name": "UBL 1-3", "date": "11/23/2025"},
    # {"id": "5669429", "name": "UBL 1-4", "date": "11/29/2025"},
    # {"id": "7826390", "name": "UBL 1-5", "date": "11/30/2025"},
    # {"id": "1562825", "name": "UBL 2-1", "date": "2/7/2026"},
    # {"id": "8943892", "name": "UBL 2-2", "date": "2/8/2026"},
    # {"id": "7439204", "name": "UBL 2-3", "date": "2/21/2026"},
    # {"id": "9405009", "name": "UBL 2-4", "date": "2/22/2026"},
    # {"id": "5548854", "name": "UBL 3-1", "date": "3/22/2026"}
]

# Set a concurrency limit.
MAX_CONCURRENT_REQUESTS = 3 

async def fetch_ubl_player_simple(semaphore, session, player_dict, tid, name, date, base_url, profile_url):
    """Async worker to fetch a single player using a shared session (Cookie Spoofing)."""
    current_id = player_dict['id']
    dropdown_txt = player_dict['dropdown_text']

    async with semaphore:
        await asyncio.sleep(random.uniform(0.2, 0.7))
        
        # 💉 CONSTRUCT THE EXACT COOKIE
        cookie_string = f"TCG{tid}=ShikibetsuNo={current_id}&Visitor="
        
        headers = {
            'User-Agent': 'Mozilla/5.0', 
            'Referer': base_url,
            'Cookie': cookie_string
        }
        
        try:
            async with session.get(profile_url, headers=headers) as profile_resp:
                html_bytes = await profile_resp.read()
                
            soup = BeautifulSoup(html_bytes, 'html.parser')

            found_name = None
            target_str = f"'{current_id}'"
            target_row = soup.find('tr', onclick=lambda x: x and target_str in x)

            if target_row:
                cells = target_row.find_all('td')
                if len(cells) >= 2:
                    found_name = cells[1].get_text(strip=True)
                    print("found name:", found_name)

            if found_name:
                return [dropdown_txt, found_name, current_id, tid, name, date]
            
            print(f"⚠️ Name not found for ID {dropdown_txt} (Player ID: {current_id})")

            debug_filename = f"debug_tid{tid}_player{current_id}.html"
            with open(debug_filename, "wb") as f:
                f.write(html_bytes)
            
            return None

        except Exception as e:
            print(f"⚠️ Error on ID {dropdown_txt}: {e}")
            return None

async def fetch_ubl_player_real(semaphore, player_dict, tid, name, date, base_url, login_url, profile_url):
    """Async worker that performs a full real login to capture backend ASP cookies."""
    current_id = player_dict['id']
    dropdown_txt = player_dict['dropdown_text']

    async with semaphore:
        await asyncio.sleep(random.uniform(0.2, 0.7))
        
        headers = {'User-Agent': 'Mozilla/5.0', 'Referer': base_url}
        
        # Open a NEW session for this player to isolate the ASPSESSIONID
        async with aiohttp.ClientSession(headers=headers) as session:
            try:
                # 1. Visit base to get initial cookie
                async with session.get(base_url) as resp:
                    await resp.read() 
                
                # 2. Submit exact form payload
                payload = {
                    "tid": tid,
                    "MMP": "",
                    "OnlineResult": "2",
                    "Dummy": "",
                    "ShikibetsuNo": current_id,
                    "flu": "",
                    "SelectShikibetsuNo": current_id
                }
                
                session.headers.update({'Referer': base_url})
                async with session.post(login_url, data=payload) as post_resp:
                    await post_resp.read()

                # 3. Fetch profile
                session.headers.update({'Referer': login_url})
                async with session.get(profile_url) as profile_resp:
                    html_bytes = await profile_resp.read()
                    
                soup = BeautifulSoup(html_bytes, 'html.parser')

                found_name = None
                target_str = f"'{current_id}'"
                target_row = soup.find('tr', onclick=lambda x: x and target_str in x)

                if target_row:
                    cells = target_row.find_all('td')
                    if len(cells) >= 2:
                        found_name = cells[1].get_text(strip=True)
                        print("found name:", found_name)

                if found_name:
                    return [dropdown_txt, found_name, current_id, tid, name, date]
                
                print(f"⚠️ Name not found for ID {dropdown_txt} (Player ID: {current_id})")

                debug_filename = f"debug_tid{tid}_player{current_id}.html"
                with open(debug_filename, "wb") as f:
                    f.write(html_bytes)
                
                return None

            except Exception as e:
                print(f"⚠️ Error on ID {dropdown_txt}: {e}")
                return None


async def scrape_ubl_tournament(tid, name, date, csv_filename):
    print(f"📊 Scraping Tournament: {name} ({date}) with ID: {tid}")

    BASE_URL = f"https://tcg.sfc-jpn.jp/loginnum.asp?tid={tid}&MMP=&flu=&Exclusive=0"
    PROFILE_URL = f"https://tcg.sfc-jpn.jp/tour.asp?tid={tid}&kno=1&znt=1&MMP=&flu=&Exclusive=0"
    LOGIN_ACTION_URL = "https://tcg.sfc-jpn.jp/login_bin.asp"
    
    dropdown_results = []
    
    # 1. Fetch the dropdown list using a temporary session
    async with aiohttp.ClientSession() as session:
        try:
            async with session.get(BASE_URL) as resp:
                html_bytes = await resp.read()
                soup = BeautifulSoup(html_bytes, 'html.parser')

                select = soup.find('select', attrs={'name': 'SelectShikibetsuNo'})
                if not select:
                    print("❌ Error: Dropdown not found.")
                    return

                for opt in select.find_all('option'):
                    val = opt.get('value')
                    text = opt.get_text(strip=True)
                    if val and val.strip():
                        dropdown_results.append({'id': val, 'dropdown_text': text})
        except Exception as e:
            print(f"❌ Connection Error: {e}")
            return

    print(f"Found {len(dropdown_results)} players in dropdown. Starting async extraction (MODE: {MODE})...")

    # 2. Set up the semaphore
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    
    # 3. Route tasks based on the MODE flag
    if MODE == "simple":
        async with aiohttp.ClientSession() as shared_session:
            tasks = [
                fetch_ubl_player_simple(semaphore, shared_session, p, tid, name, date, BASE_URL, PROFILE_URL) 
                for p in dropdown_results
            ]
            results = await asyncio.gather(*tasks)
            
    elif MODE == "real":
        tasks = [
            fetch_ubl_player_real(semaphore, p, tid, name, date, BASE_URL, LOGIN_ACTION_URL, PROFILE_URL) 
            for p in dropdown_results
        ]
        results = await asyncio.gather(*tasks)
        
    else:
        print(f"❌ Error: Unknown MODE '{MODE}'. Please set MODE to 'simple' or 'real'.")
        return

    # Filter out None values and save to CSV
    players_in_tournament = [res for res in results if res is not None]

    with open(csv_filename, 'a', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        if csvfile.tell() == 0:
            writer.writerow(['Dropdown Text', 'Player Name', 'Player ID', 'Tournament ID', 'Tournament Name', 'Tournament Date'])
        writer.writerows(players_in_tournament)
        print(f"💾 Saved {len(players_in_tournament)} records to {csv_filename}")


def clean_csv_line(line: str) -> str:
    parts = line.split(',')
    
    if len(parts) <= 7:
        return line
        
    if len(parts) == 8:
        if parts[4].strip().isdigit(): 
            parts[2] = f'"{parts[2]},{parts[3]}"'
            del parts[3]
        elif parts[6].strip().isdigit():
            parts[4] = f'"{parts[4]},{parts[5]}"'
            del parts[5]
            
    elif len(parts) == 9:
        parts[5] = f'"{parts[5]},{parts[6]}"'
        del parts[6]
        parts[2] = f'"{parts[2]},{parts[3]}"'
        del parts[3]
        
    return ','.join(parts)


async def scrape_pbl_mbl_tournament(tid, name, date, csv_filename):
    CSV_URL = f"https://tcg.sfc-jpn.jp/tourcsvdl.asp?tid={tid}&kno=3&blk=&znt=0&Sort=Table&Order=Asc&CharCode=UTF-8&MMP=1&flu=&CurScr=0"

    print(f"\nFetching and cleaning CSV data from {CSV_URL}...")
    try:
        async with aiohttp.ClientSession() as session:
            async with session.get(CSV_URL) as response:
                response.raise_for_status()
                raw_text = await response.text(encoding='utf-8')
        
        cleaned_lines = [clean_csv_line(line) for line in raw_text.splitlines()]
        cleaned_text = '\n'.join(cleaned_lines)
        df = pd.read_csv(io.StringIO(cleaned_text))
        
    except Exception as e:
        print(f"Error fetching or parsing the CSV: {e}")
        return

    if not {'No.', 'Your name'}.issubset(df.columns):
        print("Error: The CSV does not contain the required 'No.' and 'Your name' columns.")
        return

    csv_updates_lookup = dict(zip(df['No.'], df['Your name']))

    players_in_tournament = [
        [index, v, k, tid, name, date] 
        for index, (k, v) in enumerate(csv_updates_lookup.items(), start=1)
    ]

    with open(csv_filename, 'a', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        if csvfile.tell() == 0:
            writer.writerow(['Dropdown Text', 'Player Name', 'Player ID', 'Tournament ID', 'Tournament Name', 'Tournament Date'])
        writer.writerows(players_in_tournament)
        print(f"💾 Saved {len(players_in_tournament)} records to {csv_filename}")


async def scrape_tournament(tid, name, date, csv_filename):
    if "UBL" in name:
        print(f"🔍 Detected UBL tournament type for {name}. Using UBL scraper.")
        await scrape_ubl_tournament(tid, name, date, csv_filename)
    elif "PBL" in name or "MBL" in name:
        print(f"🔍 Detected PBL/MBL tournament type for {name}. Using PBL/MBL scraper.")
        await scrape_pbl_mbl_tournament(tid, name, date, csv_filename)
    else:
        print(f"⚠️ No scraper defined for tournament type: {name}")


async def run_scraper():
    print("--- 🚀 Starting Optimized Async Scraper ---")
    
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    csv_filename = f"tournament_players_{timestamp}.csv"
    
    for tournament in TOURNAMENTS:
        print(f"\n🎯 Processing Tournament: {tournament['name']} ({tournament['date']})")
        await scrape_tournament(tournament['id'], tournament['name'], tournament['date'], csv_filename)

if __name__ == "__main__":
    await run_scraper()

--- 🚀 Starting Optimized Async Scraper ---

🎯 Processing Tournament: UBL 1-2 (11/15/2025)
🔍 Detected UBL tournament type for UBL 1-2. Using UBL scraper.
📊 Scraping Tournament: UBL 1-2 (11/15/2025) with ID: 608123
Found 48 players in dropdown. Starting async extraction (MODE: real)...
⚠️ Name not found for ID ph07229353 (Player ID: 12)
⚠️ Name not found for ID ph09812880 (Player ID: 39)
⚠️ Name not found for ID ph10141228 (Player ID: 35)
⚠️ Name not found for ID ph16366228 (Player ID: 23)
⚠️ Name not found for ID ph14812893 (Player ID: 13)
⚠️ Name not found for ID ph16974102 (Player ID: 22)
⚠️ Name not found for ID ph18698143 (Player ID: 25)
⚠️ Name not found for ID ph22460851 (Player ID: 3)
⚠️ Name not found for ID ph28389633 (Player ID: 45)
⚠️ Name not found for ID ph28446919 (Player ID: 17)
⚠️ Name not found for ID ph29622878 (Player ID: 4)
⚠️ Name not found for ID ph29429499 (Player ID: 5)
⚠️ Name not found for ID ph30757422 (Player ID: 33)
⚠️ Name not found for ID ph30906732 (Playe

CancelledError: 